In [ ]:
# 1) GPU check + package installs
import sys
print(sys.version)

!nvidia-smi
!pip -q install pretty_midi mido tqdm

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false
# 2) All imports + config values (copied from src/config.py)
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42
MIDI_MIN_PITCH = 21
MIDI_MAX_PITCH = 108
PITCH_DIM = MIDI_MAX_PITCH - MIDI_MIN_PITCH + 1

STEPS_PER_BAR = 16
BEATS_PER_BAR = 4
STEPS_PER_BEAT = STEPS_PER_BAR // BEATS_PER_BAR
DEFAULT_TEMPO_BPM = 120.0

SEQUENCE_LENGTH = 256
WINDOW_STEP = 128

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
VALIDATION_SPLIT = 0.2
NUM_WORKERS = 0

VAE_HIDDEN_SIZE = 256
VAE_NUM_LAYERS = 2
VAE_LATENT_DIM = 64
VAE_DROPOUT = 0.2
VAE_BETA = 1e-3
VAE_NUM_SAMPLES = 8
VAE_INTERPOLATION_STEPS = 10

# Task 2 is multi-genre: attach at least two genre datasets in Kaggle.
GENRE_DATASET_CANDIDATES = {
    'classical_maestro': [
        Path('/kaggle/input/datasets/jackvial/themaestrodatasetv2'),
        Path('/kaggle/input/themaestrodatasetv2'),
    ],
    'multi_genre_lakh': [
        Path('/kaggle/input/datasets/imsparsh/lakh-midi-clean'),
        Path('/kaggle/input/lakh-midi-clean'),
        Path('/kaggle/input/lakh-midi-dataset'),
        Path('/kaggle/input/lakh-midi'),
        Path('/kaggle/input/lmd-full'),
        Path('/kaggle/input/lmd-matched'),
    ],
    'jazz_groove': [
        Path('/kaggle/input/groove-midi-dataset'),
        Path('/kaggle/input/groove-midi'),
        Path('/kaggle/input/groove'),
    ],
}

ACTIVE_GENRE_ROOTS = {}
for genre_name, candidates in GENRE_DATASET_CANDIDATES.items():
    for candidate in candidates:
        if candidate.exists():
            ACTIVE_GENRE_ROOTS[genre_name] = candidate
            break

if len(ACTIVE_GENRE_ROOTS) < 2:
    raise RuntimeError(
        'Task 2 requires multi-genre training. Attach at least two datasets (for example MAESTRO + Lakh or MAESTRO + Groove) and update GENRE_DATASET_CANDIDATES in Cell 2 if needed.'
    )

OUTPUT_ROOT = Path('/kaggle/working/outputs')
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
GENERATED_MIDI_DIR = OUTPUT_ROOT / 'generated_midis'

for directory in [OUTPUT_ROOT, CHECKPOINTS_DIR, PLOTS_DIR, GENERATED_MIDI_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Active genre roots:')
for genre_name, genre_root in ACTIVE_GENRE_ROOTS.items():
    print(f'  - {genre_name}: {genre_root}')

CODE_INPUT_CANDIDATES = [
    Path('/kaggle/input/music-generation-unsupervised-task2-src'),
    Path('/kaggle/input/music-generation-unsupervised-task1-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task2-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task1-src'),
]

code_input_root = None
for candidate in CODE_INPUT_CANDIDATES:
    if candidate.exists():
        code_input_root = candidate
        break

if code_input_root is None:
    raise FileNotFoundError(
        'Code dataset path not found. Run "!ls /kaggle/input" and update CODE_INPUT_CANDIDATES.'
    )

if (code_input_root / 'music-generation-unsupervised').exists():
    code_input_root = code_input_root / 'music-generation-unsupervised'

PROJECT_ROOT = Path('/kaggle/working/music-generation-unsupervised')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(code_input_root, PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Code input used:', code_input_root)
print('Project root exists:', PROJECT_ROOT.exists())

In [ ]:
# pyright: reportMissingImports=false
# 3) Data loading and preprocessing (multi-genre aggregation)
from torch.utils.data import DataLoader

from src.preprocessing.midi_parser import discover_midi_files
from src.preprocessing.piano_roll import load_windowed_piano_rolls
from src.training.train_ae import PianoRollWindowDataset, split_midi_files

MAX_FILES_PER_GENRE = 120  # Set to an integer to balance sources (for example 2000)

all_midi_files = []
for genre_name, genre_root in ACTIVE_GENRE_ROOTS.items():
    genre_files = discover_midi_files(genre_root)
    if MAX_FILES_PER_GENRE is not None:
        genre_files = genre_files[:MAX_FILES_PER_GENRE]
    print(f'{genre_name}: {len(genre_files)} files from {genre_root}')
    all_midi_files.extend(genre_files)

all_midi_files = sorted(set(all_midi_files))
print(f'Total unique MIDI files across genres: {len(all_midi_files)}')

train_files, validation_files = split_midi_files(
    all_midi_files,
    validation_split=VALIDATION_SPLIT,
    seed=SEED,
    split_name='task2_multigenre',
)

train_windows = load_windowed_piano_rolls(train_files, sequence_length=SEQUENCE_LENGTH, step_size=WINDOW_STEP)
validation_windows = load_windowed_piano_rolls(validation_files, sequence_length=SEQUENCE_LENGTH, step_size=WINDOW_STEP)

train_dataset = PianoRollWindowDataset(train_windows)
validation_dataset = PianoRollWindowDataset(validation_windows)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f'Train windows: {len(train_dataset)}')
print(f'Validation windows: {len(validation_dataset)}')

In [ ]:
# 4) Model definition (from src/models/vae.py)
from src.models.vae import MusicVAE

model = MusicVAE(
    input_dim=PITCH_DIM,
    hidden_size=VAE_HIDDEN_SIZE,
    num_layers=VAE_NUM_LAYERS,
    latent_dim=VAE_LATENT_DIM,
    dropout=VAE_DROPOUT,
).to(DEVICE)

model

In [ ]:
# 5) Training loop (from src/training/train_vae.py)
from src.training.train_vae import train_vae

results = train_vae(
    train_loader=train_loader,
    validation_loader=validation_loader,
    model=model,
    beta=VAE_BETA,
)

print('Best checkpoint saved at:', results['checkpoint_path'])
print('Last train total loss:', results['train_total_losses'][-1])
print('Last validation total loss:', results['validation_total_losses'][-1])

In [ ]:
# 6) Loss and interpolation plots save
loss_plot_path = PLOTS_DIR / 'task2_loss_curve.png'

plt.figure(figsize=(10, 5))
plt.plot(results['train_total_losses'], label='Train Total')
plt.plot(results['validation_total_losses'], label='Validation Total')
plt.plot(results['validation_reconstruction_losses'], label='Validation Recon', alpha=0.8)
plt.plot(results['validation_kl_losses'], label='Validation KL', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Task 2 - VAE Loss Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(loss_plot_path, dpi=200)
plt.show()

print('Saved:', loss_plot_path)

In [ ]:
# 7) Music generation and latent interpolation
from src.generation.generate_music import build_task2_latent_interpolation, generate_task2_samples

generated_paths = generate_task2_samples(
    checkpoint_path=results['checkpoint_path'],
    num_samples=VAE_NUM_SAMPLES,
    output_dir=GENERATED_MIDI_DIR,
    tempo_bpm=DEFAULT_TEMPO_BPM,
    latent_dim=VAE_LATENT_DIM,
)

# Start with random-latent interpolation.
interpolation_rolls = build_task2_latent_interpolation(
    checkpoint_path=results['checkpoint_path'],
    num_steps=VAE_INTERPOLATION_STEPS,
    sequence_length=SEQUENCE_LENGTH,
    latent_dim=VAE_LATENT_DIM,
)
interpolation_panel = np.concatenate(interpolation_rolls, axis=1).astype(np.float32)

panel_spread = float(np.percentile(interpolation_panel, 99) - np.percentile(interpolation_panel, 1))
interpolation_source = 'random-latent'

# If random interpolation is too flat, use latents encoded from real validation samples.
if panel_spread < 1e-5 or float(interpolation_panel.max()) < 1e-4:
    batch = next(iter(validation_loader))
    if batch.size(0) >= 2:
        with torch.no_grad():
            mu, _ = model.encode(batch[:2].to(DEVICE))
            weights = torch.linspace(0.0, 1.0, steps=VAE_INTERPOLATION_STEPS, device=DEVICE).unsqueeze(1)
            latent_path = mu[0:1] * (1.0 - weights) + mu[1:2] * weights
            decoded_path = model.decode(latent_path, sequence_length=SEQUENCE_LENGTH)
            interpolation_rolls = decoded_path.detach().cpu().numpy()
        interpolation_panel = np.concatenate(interpolation_rolls, axis=1).astype(np.float32)
        interpolation_source = 'data-anchored'

# Focus on the most active pitches for clearer visualization.
pitch_energy = interpolation_panel.sum(axis=0)
if np.count_nonzero(pitch_energy) > 0:
    top_pitch_count = min(48, interpolation_panel.shape[1])
    top_pitch_indices = np.argsort(pitch_energy)[-top_pitch_count:]
    interpolation_panel_vis = interpolation_panel[:, top_pitch_indices]
else:
    interpolation_panel_vis = interpolation_panel

# Contrast enhancement for low-amplitude probability maps.
panel_for_plot = np.log1p(interpolation_panel_vis * 30.0)
vmin = float(np.percentile(panel_for_plot, 1))
vmax = float(np.percentile(panel_for_plot, 99))
if vmax <= vmin:
    vmax = vmin + 1e-3

interpolation_plot_path = PLOTS_DIR / 'task2_latent_interpolation.png'
plt.figure(figsize=(14, 6))
plt.imshow(
    panel_for_plot.T,
    aspect='auto',
    origin='lower',
    cmap='magma',
    vmin=vmin,
    vmax=vmax,
)
plt.xlabel('Interpolated Time Steps')
plt.ylabel('Selected Active Pitch Bins')
plt.title('Task 2 - Latent Interpolation Piano-Roll')
plt.tight_layout()
plt.savefig(interpolation_plot_path, dpi=200)
plt.show()

active_ratio = float((interpolation_panel > 0.01).mean())
print('Interpolation source:', interpolation_source)
print('Interpolation stats:', f'min={interpolation_panel.min():.4f}', f'max={interpolation_panel.max():.4f}', f'mean={interpolation_panel.mean():.4f}')
print('Interpolation active ratio (>0.01):', f'{active_ratio:.4f}')
print('Saved:', interpolation_plot_path)
for path in generated_paths:
    print(path)

In [ ]:
# 8) Summary cell printing all deliverables
expected_files = [
    PLOTS_DIR / 'task2_loss_curve.png',
    PLOTS_DIR / 'task2_latent_interpolation.png',
] + [GENERATED_MIDI_DIR / f'task2_sample_{index}.mid' for index in range(1, 9)]

print('Task 2 Deliverables Summary')
for file_path in expected_files:
    status = 'OK' if file_path.exists() else 'MISSING'
    print(f'[{status}] {file_path}')